# 03 — Representaciones en frecuencia (Transformada de Fourier)

Calculamos la DFT / FFT de la señal de voz y construimos representaciones tiempo-frecuencia: espectro, espectrograma y espectrograma de Mel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from pathlib import Path

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data')

# Usar la señal preprocesada si existe, o la original
ruta = DATA_DIR / 'procesado.wav'
if not ruta.exists():
    ruta = sorted(DATA_DIR.glob('*.wav'))[0]

y, sr = librosa.load(ruta, sr=None)
print(f'Señal: {ruta.name} | sr={sr} Hz | dur={len(y)/sr:.2f}s')

## 1. DFT / FFT — Espectro de amplitud

In [ ]:
N = len(y)
Y = np.fft.rfft(y)           # Solo la mitad positiva (señal real)
freqs = np.fft.rfftfreq(N, d=1/sr)
magnitud = np.abs(Y)
magnitud_db = 20 * np.log10(magnitud + 1e-10)

fig, axes = plt.subplots(2, 1)
axes[0].plot(freqs, magnitud, linewidth=0.6, color='steelblue')
axes[0].set_title('Espectro de amplitud (FFT)')
axes[0].set_ylabel('|X(f)|')
axes[0].set_xlim(0, sr/2)

axes[1].plot(freqs, magnitud_db, linewidth=0.6, color='darkorange')
axes[1].set_title('Espectro de amplitud (dB)')
axes[1].set_ylabel('|X(f)| [dB]')
axes[1].set_xlabel('Frecuencia [Hz]')
axes[1].set_xlim(0, sr/2)

plt.tight_layout()
plt.savefig('../figuras/03_espectro_fft.png', dpi=150)
plt.show()

## 2. Frecuencia fundamental y armónicos

In [ ]:
# Buscar los 5 picos más prominentes entre 80 Hz y 500 Hz
mascara = (freqs >= 80) & (freqs <= 500)
idx_f0 = np.argmax(magnitud[mascara])
f0 = freqs[mascara][idx_f0]
print(f'Frecuencia fundamental estimada (F0): {f0:.1f} Hz')
print(f'Primeros armónicos esperados: {[round(f0*k,1) for k in range(2,6)]}')

## 3. Espectrograma STFT

In [ ]:
n_fft = 2048
hop_length = 512

D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
D_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

fig, ax = plt.subplots(figsize=(14, 5))
img = librosa.display.specshow(D_db, sr=sr, hop_length=hop_length,
                                x_axis='time', y_axis='hz', ax=ax)
fig.colorbar(img, ax=ax, format='%+2.0f dB')
ax.set_title('Espectrograma STFT')
ax.set_ylim(0, 8000)
plt.tight_layout()
plt.savefig('../figuras/03_espectrograma_stft.png', dpi=150)
plt.show()

## 4. Espectrograma de Mel

In [ ]:
n_mels = 128
S_mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft,
                                        hop_length=hop_length, n_mels=n_mels)
S_mel_db = librosa.power_to_db(S_mel, ref=np.max)

fig, ax = plt.subplots(figsize=(14, 5))
img = librosa.display.specshow(S_mel_db, sr=sr, hop_length=hop_length,
                                x_axis='time', y_axis='mel', ax=ax)
fig.colorbar(img, ax=ax, format='%+2.0f dB')
ax.set_title('Espectrograma de Mel')
plt.tight_layout()
plt.savefig('../figuras/03_espectrograma_mel.png', dpi=150)
plt.show()

## 5. MFCCs (Mel-Frequency Cepstral Coefficients)

In [ ]:
n_mfcc = 13
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
                              n_fft=n_fft, hop_length=hop_length)

fig, ax = plt.subplots(figsize=(14, 4))
img = librosa.display.specshow(mfcc, x_axis='time', ax=ax)
fig.colorbar(img, ax=ax)
ax.set_title('MFCCs')
ax.set_ylabel('Coeficiente MFCC')
plt.tight_layout()
plt.savefig('../figuras/03_mfcc.png', dpi=150)
plt.show()

print(f'Shape MFCCs: {mfcc.shape}  ({n_mfcc} coeficientes × {mfcc.shape[1]} frames)')